# 第39章 点图（pointplot）

用点和连接线比较类别估计值，突出差异方向与交互模式。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较多个类别或两个因素下的均值趋势，不需要柱形面积。

## 数据结构

一至两个分类变量和一个数值变量。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 dodge=0.25 改为 dodge=0，观察分组错位与重叠显示的差异
2. 修改 markers 参数从 ["o", "s", "^"] 为 ["D", "v", "p"]，对比不同标记形状的区分度
3. 调整 linestyles 从 ["-", "--", ":"] 为全部 "-"，说明线型对多组区分的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(data=orders, x="category", y="order_value", errorbar=("ci", 90), color="#1a73e8", ax=ax)
ax.set(title="品类客单价点估计", xlabel="品类", ylabel="平均客单价（元）")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.pointplot(data=orders, x="category", y="order_value", hue="channel", dodge=0.25, markers=["o", "s", "^"], linestyles=["-", "--", ":"], errorbar=None, palette="colorblind", ax=ax)
ax.set(title="渠道与品类客单价模式", xlabel="品类", ylabel="平均客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 3. 参数说明

- estimator：点估计
- errorbar：误差
- dodge：分组错位
- markers/linestyles：样式


## 4. 结果解读

读取点的位置和组间斜率；非平行线可能提示因素交互。


## 常见误区

- 把类别间连接线解释为连续时间
- 类别顺序没有业务意义
- 多组线条难以辨认


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.pointplot(data=orders, x="region", y="items", hue="satisfied", dodge=0.2, errorbar=None, palette=["#188038", "#f9ab00"], ax=ax)
ax.set(title="区域购买件数与评价", xlabel="区域", ylabel="平均件数")
ax.legend(title="评价", frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

用点和连接线比较类别估计值，突出差异方向与交互模式。


### 你已经掌握

- 判断点图（pointplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较多个类别或两个因素下的均值趋势，不需要柱形面积。 |
| 数据结构 | 一至两个分类变量和一个数值变量。 |
| 结果解读 | 读取点的位置和组间斜率；非平行线可能提示因素交互。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `estimator` | 点估计 |
| `errorbar` | 误差 |
| `dodge` | 分组错位 |
| `markers/linestyles` | 样式 |


### 需要注意

- 把类别间连接线解释为连续时间
- 类别顺序没有业务意义
- 多组线条难以辨认


### 完成检查

- [ ] 能判断什么问题适合使用点图（pointplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
